# Lab03 — Harita ve Benzinlik Konumları (Ara Çalışma)

Bu notebook'da:
1. İstanbul haritası
2. Güncel benzinlik konumları (OpenStreetMap Overpass API)
3. Harita üzerinde işaretleme (folium)

Sonrasında bu kodlar Lab03_Gas_Station_Location.ipynb'ye taşınacak.

In [40]:
# Gerekli kütüphaneler (folium harita için, requests Overpass API için)
import requests
import pandas as pd
import folium
from typing import List, Dict, Any

## 1. İstanbul sınırları ve benzinlik verisi (OpenStreetMap)

OpenStreetMap'te `amenity=fuel` ile benzinlikler işaretlidir. Overpass API ile İstanbul bölgesini sorguluyoruz (node + way + relation; sonra koordinat deduplication).

**İstanbul bbox (geniş):** lat 40.80–41.42, lon 28.45–29.65 (Silivri–Şile–Adalar dahil, gözden kaçan kalmasın)

In [41]:
# İstanbul bbox: güney-kuzey-batı-doğu (min_lat, max_lat, min_lon, max_lon)
# Biraz geniş tutuyoruz (Silivri, Şile, Adalar, Gebze yönü) — gözden kaçan kalmasın
ISTANBUL_BBOX = (40.80, 41.42, 28.45, 29.65)

# Birden fazla Overpass sunucusu — biri 504 verirse diğeri denenir
OVERPASS_SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

def fetch_gas_stations_osm(bbox: tuple, timeout_sec: int = 90) -> List[Dict[str, Any]]:
    """Overpass API ile bbox içindeki benzinlikleri (amenity=fuel) çeker.
    node + way + relation hepsi alınır. 504/timeout olursa sırayla diğer sunucular denenir.
    """
    min_lat, max_lat, min_lon, max_lon = bbox
    query = f"""
    [out:json][timeout:45];
    (
      node["amenity"="fuel"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["amenity"="fuel"]({min_lat},{min_lon},{max_lat},{max_lon});
      relation["amenity"="fuel"]({min_lat},{min_lon},{max_lat},{max_lon});
    );
    out center;
    """
    last_error = None
    for url in OVERPASS_SERVERS:
        try:
            r = requests.get(url, params={"data": query}, timeout=timeout_sec)
            r.raise_for_status()
            return r.json().get("elements", [])
        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 504:
                last_error = e
                continue
            raise
        except requests.exceptions.Timeout:
            last_error = None
            continue
    if last_error:
        raise last_error
    raise requests.exceptions.Timeout("Tüm Overpass sunucuları zaman aşımına uğradı.")

In [42]:
# Benzinlikleri çek ve koordinatlara dönüştür (node: lat/lon; way/relation: center veya bounds)
raw = fetch_gas_stations_osm(ISTANBUL_BBOX)

def element_to_coords(el: dict) -> tuple:
    """node → (lat,lon); way → center; relation → center veya bounds merkezi."""
    if el.get("type") == "node":
        return (el["lat"], el["lon"])
    if el.get("type") == "way" and "center" in el:
        return (el["center"]["lat"], el["center"]["lon"])
    if el.get("type") == "relation":
        if "center" in el:
            return (el["center"]["lat"], el["center"]["lon"])
        if "bounds" in el:
            b = el["bounds"]
            return ((b["minlat"] + b["maxlat"]) / 2, (b["minlon"] + b["maxlon"]) / 2)
    return None

rows = []
for el in raw:
    coords = element_to_coords(el)
    if coords is None:
        continue
    name = (el.get("tags") or {}).get("name", "")
    brand = (el.get("tags") or {}).get("brand", "")
    rows.append({"lat": coords[0], "lon": coords[1], "name": name, "brand": brand})

# Aynı yerde birden fazla kayıt olabilir (node+way aynı benzinlik); koordinatı yuvarlayıp tekilleştir
def dedup_by_rounded_coords(rows: list, decimals: int = 5) -> list:
    seen = set()
    out = []
    for r in rows:
        key = (round(r["lat"], decimals), round(r["lon"], decimals))
        if key in seen:
            continue
        seen.add(key)
        out.append(r)
    return out

rows = dedup_by_rounded_coords(rows)
df_gas = pd.DataFrame(rows)
print(f"Toplam benzinlik sayısı (dedup sonrası): {len(df_gas)}")
df_gas.head(10)

## 2. İstanbul haritası ve benzinliklerin işaretlenmesi

Harita merkezi: İstanbul (lat 41.01, lon 28.95). Benzinlikler kırmızı daire ile gösteriliyor.

In [43]:
# İstanbul merkez koordinatları
ISTANBUL_CENTER = [41.01, 28.95]

m = folium.Map(location=ISTANBUL_CENTER, zoom_start=10, tiles="OpenStreetMap")

for _, row in df_gas.iterrows():
    popup = f"{row['name'] or 'Benzinlik'} ({row['brand'] or '-'})"
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.7,
        popup=popup,
        weight=1,
    ).add_to(m)

m

In [24]:
# Haritayı HTML olarak kaydetmek isterseniz (opsiyonel)
# m.save("istanbul_benzinlikler.html")
# print("Harita istanbul_benzinlikler.html olarak kaydedildi.")

### Veri kaynağı (Lab raporu için)

- **Benzinlik konumları:** OpenStreetMap (OSM), Overpass API üzerinden.
- **Sorgu:** `amenity=fuel` (İstanbul bbox içinde).
- **Toplam:** Yukarıdaki sayı kadar nokta (node + way center). Lab en az 200 istasyon istiyor; OSM verisi buna yakın veya üzerinde olabilir.
- **API:** https://overpass-api.de/api/interpreter — ücretsiz, kayıt yok.

In [25]:
# İsterseniz benzinlik listesini CSV olarak kaydedebilirsiniz (ana lab'da kullanmak için)
# df_gas.to_csv("istanbul_benzinlikler_osm.csv", index=False)
# print("Kaydedildi: istanbul_benzinlikler_osm.csv")

---
## 3. İBB Trafik Verileri

**Kaynak:** [İBB Açık Veri Portalı](https://data.ibb.gov.tr/) (data.ibb.gov.tr)

### Bulunan trafik veri setleri

| Veri seti | Açıklama | Format | Kullanım |
|-----------|----------|--------|----------|
| **Saatlik Trafik Yoğunluk Veri Seti** | İstanbul coğrafi parçalar (geohash) bazında saatlik araç sayısı, ortalama/max/min hız | CSV (aylık dosyalar) | **Ana kaynak** — Lab ile uyumlu (lokasyon, araç sayısı, hız) |
| Trafik Yoğunluk Haritası | Canlı harita uygulaması | HTML link (uym.ibb.gov.tr) | Ham veri değil, sadece harita |
| Ulaşım Yönetim Merkezi Trafik Duyuru Verisi | Trafik olay/duyuru kayıtları (tür, enlem, boylam, tarih) | CSV | Talep skorunda ikincil kullanılabilir |

### Saatlik Trafik Yoğunluk Veri Seti detayı

- **Portal sayfası:** https://data.ibb.gov.tr/dataset/hourly-traffic-density-data-set  
- **CKAN API:** `package_show?id=hourly-traffic-density-data-set`  
- **Kapsam:** Ocak 2020 – Ocak 2025, aylık CSV (61 dosya).  
- **İçerik (İBB açıklaması):** İstanbul coğrafi olarak eşit parçalara bölünmüş; her parçada tekil araç sayısı, ortalama hız, maksimum hız, minimum hız.  
- **Lisans:** İstanbul Büyükşehir Belediyesi Açık Veri Lisansı.

Aşağıda bu veri setinden API ile kaynak listesi alınıp, seçilen bir ayın CSV’si indirilip yükleniyor.

In [26]:
# İBB CKAN API — Saatlik Trafik Yoğunluk veri seti
IBB_CKAN = "https://data.ibb.gov.tr/api/3/action"
PACKAGE_ID = "hourly-traffic-density-data-set"

def ibb_package_show(package_id: str) -> dict:
    """Veri seti bilgisi ve kaynak listesini döner."""
    r = requests.get(f"{IBB_CKAN}/package_show", params={"id": package_id})
    r.raise_for_status()
    return r.json()["result"]

pkg = ibb_package_show(PACKAGE_ID)
print("Veri seti:", pkg["title"])
print("Kaynak sayısı:", pkg["num_resources"])
print("\nSon birkaç kaynak (aylık CSV):")
for res in pkg["resources"][-5:]:
    print(f"  - {res['name']}: {res.get('url', '')[:80]}...")

Veri seti: Hourly Traffic Density Data Set
Kaynak sayısı: 61

Son birkaç kaynak (aylık CSV):
  - Eylül 2024 Trafik Yoğunluk Verisi: https://data.ibb.gov.tr/dataset/3ee6d744-5da2-40c8-9cd6-0e3e41f1928f/resource/91...
  - Ekim 2024 Trafik Yoğunluk Verisi: https://data.ibb.gov.tr/dataset/3ee6d744-5da2-40c8-9cd6-0e3e41f1928f/resource/d2...
  - Kasım 2024 Trafik Yoğunluk Verisi: https://data.ibb.gov.tr/dataset/3ee6d744-5da2-40c8-9cd6-0e3e41f1928f/resource/be...
  - Aralık 2024 Trafik Yoğunluk Verisi: https://data.ibb.gov.tr/dataset/3ee6d744-5da2-40c8-9cd6-0e3e41f1928f/resource/76...
  - Ocak 2025 Trafik Yoğunluk Verisi: https://data.ibb.gov.tr/dataset/3ee6d744-5da2-40c8-9cd6-0e3e41f1928f/resource/57...


In [27]:
# Belirli bir ayın CSV'sini indir (örnek: Ekim 2024 — lab'daki örnek tarihe yakın)
def download_ibb_traffic_month(package_result: dict, year: int, month: int) -> pd.DataFrame:
    """Yıl ve ay vererek o aya ait IBB trafik yoğunluk CSV'sini indirir ve DataFrame döner."""
    suffix = f"traffic_density_{year}{month:02d}.csv"
    res = None
    for r in package_result["resources"]:
        url = r.get("url") or ""
        if suffix in url:
            res = r
            break
    if res is None:
        raise ValueError(f"{year}-{month:02d} için kaynak bulunamadı. Mevcut kaynaklardan birini seçin.")
    url = res["url"]
    print(f"İndiriliyor: {res['name']}")
    df = pd.read_csv(url)
    print(f"Satır: {len(df):,}, Sütunlar: {list(df.columns)}")
    return df

# Örnek: Ekim 2024 (tek bir ay ~100–140 MB civarı olabilir; ilk çalıştırmada süre alabilir)
df_traffic = download_ibb_traffic_month(pkg, 2024, 10)
df_traffic.head()

İndiriliyor: Ekim 2024 Trafik Yoğunluk Verisi
Satır: 1,769,643, Sütunlar: ['DATE_TIME', 'LATITUDE', 'LONGITUDE', 'GEOHASH', 'MINIMUM_SPEED', 'MAXIMUM_SPEED', 'AVERAGE_SPEED', 'NUMBER_OF_VEHICLES']


,DATE_TIME,LATITUDE,LONGITUDE,GEOHASH,MINIMUM_SPEED,MAXIMUM_SPEED,AVERAGE_SPEED,NUMBER_OF_VEHICLES
0,2024-10-01 00:00:00,41.042175,28.921509,sxk96p,6,59,33,25
1,2024-10-01 00:00:00,41.097107,28.789673,sxk3z1,9,95,47,26
2,2024-10-01 00:00:00,41.091614,28.361206,sxk1v2,55,125,84,39
3,2024-10-01 00:00:00,40.992737,28.800659,sxk3pq,32,127,62,28
4,2024-10-01 00:00:00,40.981750,28.734741,sxk3ju,7,117,50,139


In [28]:
# Sütun isimleri ve örnek değerler (IBB CSV formatı lab'daki istanbul_traffic_week ile benzer olabilir)
print("Sütunlar:", df_traffic.columns.tolist())
print("\nVeri tipleri:")
print(df_traffic.dtypes)
print("\nİlk 3 satır:")
df_traffic.head(3)

Sütunlar: ['DATE_TIME', 'LATITUDE', 'LONGITUDE', 'GEOHASH', 'MINIMUM_SPEED', 'MAXIMUM_SPEED', 'AVERAGE_SPEED', 'NUMBER_OF_VEHICLES']

Veri tipleri:
DATE_TIME              object
LATITUDE              float64
LONGITUDE             float64
GEOHASH                object
MINIMUM_SPEED           int64
MAXIMUM_SPEED           int64
AVERAGE_SPEED           int64
NUMBER_OF_VEHICLES      int64
dtype: object

İlk 3 satır:


,DATE_TIME,LATITUDE,LONGITUDE,GEOHASH,MINIMUM_SPEED,MAXIMUM_SPEED,AVERAGE_SPEED,NUMBER_OF_VEHICLES
0,2024-10-01 00:00:00,41.042175,28.921509,sxk96p,6,59,33,25
1,2024-10-01 00:00:00,41.097107,28.789673,sxk3z1,9,95,47,26
2,2024-10-01 00:00:00,41.091614,28.361206,sxk1v2,55,125,84,39


### Trafik verilerini bulduğum yerler (özet)

| Nereden | Ne |
|--------|-----|
| **[data.ibb.gov.tr](https://data.ibb.gov.tr)** | Ana portal. "Veri Setleri" veya arama ile "trafik" / "yoğunluk" yazarak ulaştım. |
| **Saatlik Trafik Yoğunluk Veri Seti** | [Doğrudan link](https://data.ibb.gov.tr/dataset/hourly-traffic-density-data-set). Aylık CSV’ler (Ocak 2020 – Ocak 2025). Araç sayısı + ortalama/max/min hız, coğrafi parça (geohash) bazında. |
| **IBB CKAN API** | `https://data.ibb.gov.tr/api/3/action/package_show?id=hourly-traffic-density-data-set` ile veri seti ve tüm resource URL’leri alındı. İndirme için `resource["url"]` kullanıldı. |
| **Trafik Duyuru Verisi** | [UYM Trafik Duyuru Verisi](https://data.ibb.gov.tr/dataset/ulasim-yonetim-merkezi-trafik-duyuru-verisi) — olay/duyuru bazlı; talep skoru için ikincil kaynak olarak kullanılabilir. |

**Eksik / sınırlar:** Saatlik veri aylık dosyalar halinde; tek bir hafta için filtreleme kod tarafında yapılmalı. Canlı/anlık API yok; sadece indirilebilir CSV. İstersen ileride Google Maps vb. ek kaynaklarla (ücretli) tamamlanabilir.

---
## 4. Kurs tarafından sağlanan trafik verisi (isteğe bağlı başlangıç)

Lab dokümanında verilen **bir haftalık saatlik sensör verisi** — İstanbul genelinde Ekim 2024, bir hafta.

| Dosya | Açıklama |
|-------|----------|
| `istanbul_traffic_week.csv` | DATE_TIME, LATITUDE, LONGITUDE, GEOHASH, MINIMUM_SPEED, MAXIMUM_SPEED, AVERAGE_SPEED, NUMBER_OF_VEHICLES (~75.000 kayıt, ~2.400 sensör) |
| `sensor_coords.csv` | Varsa: node_id (geohash), lat, long — trafik verisi ile GEOHASH üzerinden birleştirilebilir. |

Aşağıda bu CSV yükleniyor; IBB verisi ile birlikte veya tek başına kullanılabilir.

In [29]:
# Kurs repo'sundaki trafik verisi — lab/ veya Week03_NumPy_Pandas/ dizininden çalıştırılabilir
import os

_candidates = ["../istanbul_traffic_week.csv", "istanbul_traffic_week.csv", "Week03_NumPy_Pandas/istanbul_traffic_week.csv"]
TRAFFIC_WEEK_CSV = next((p for p in _candidates if os.path.isfile(p)), None)
if TRAFFIC_WEEK_CSV is None:
    TRAFFIC_WEEK_CSV = "../istanbul_traffic_week.csv"  # varsayılan

if os.path.isfile(TRAFFIC_WEEK_CSV):
    df_week = pd.read_csv(TRAFFIC_WEEK_CSV)
    df_week["DATE_TIME"] = pd.to_datetime(df_week["DATE_TIME"])
    print("istanbul_traffic_week.csv yüklendi.")
    print(f"Satır: {len(df_week):,}, Sütunlar: {list(df_week.columns)}")
    print(f"Tarih aralığı: {df_week['DATE_TIME'].min()} — {df_week['DATE_TIME'].max()}")
    print(f"Benzersiz GEOHASH (sensör): {df_week['GEOHASH'].nunique()}")
    display(df_week.head())
else:
    df_week = None
    print("Dosya bulunamadı:", TRAFFIC_WEEK_CSV)

istanbul_traffic_week.csv yüklendi.
Satır: 75,000, Sütunlar: ['DATE_TIME', 'LATITUDE', 'LONGITUDE', 'GEOHASH', 'MINIMUM_SPEED', 'MAXIMUM_SPEED', 'AVERAGE_SPEED', 'NUMBER_OF_VEHICLES']
Tarih aralığı: 2024-10-01 00:00:00 — 2024-10-07 23:00:00
Benzersiz GEOHASH (sensör): 2439


,DATE_TIME,LATITUDE,LONGITUDE,GEOHASH,MINIMUM_SPEED,MAXIMUM_SPEED,AVERAGE_SPEED,NUMBER_OF_VEHICLES
0,2024-10-01,41.119080,29.042358,sxk9uv,49,67,59,3
1,2024-10-01,41.064148,29.064331,sxk9t7,8,48,27,6
2,2024-10-01,41.091614,29.031372,sxk9u8,10,149,77,180
3,2024-10-01,41.108093,29.086304,sxk9vg,2,60,39,12
4,2024-10-01,41.113586,29.042358,sxk9uu,7,88,46,16


In [30]:
# Opsiyonel: sensor_coords.csv (GEOHASH — koordinat eşlemesi; varsa kullan)
SENSOR_COORDS_CSV = next((p for p in ["../sensor_coords.csv", "sensor_coords.csv"] if os.path.isfile(p)), None)
if SENSOR_COORDS_CSV:
    df_sensor_coords = pd.read_csv(SENSOR_COORDS_CSV)
    print("sensor_coords.csv yüklendi:", df_sensor_coords.shape)
    display(df_sensor_coords.head())
else:
    df_sensor_coords = None
    print("sensor_coords.csv bulunamadı (trafik verisinde zaten LATITUDE/LONGITUDE var).")

sensor_coords.csv bulunamadı (trafik verisinde zaten LATITUDE/LONGITUDE var).


---
## 5. Trafik verisini haritaya entegre etmek

Saatlik veri **çok zaman adımı** içeriyor; tek bir harita görselinde göstermek için **zamana göre özet** alıyoruz:

- **Lokasyon başına tek değer:** Her sensör/lokasyon (GEOHASH veya enlem–boylam) için tüm saatler üzerinden **ortalama araç sayısı** ve **ortalama hız** hesaplanır.
- **Tek harita:** Bu özet değerler haritada **tek nokta** olarak gösterilir: daire boyutu = trafik yoğunluğu (araç sayısı), renk veya ayrı bir katman = hız (isteğe bağlı).
- Böylece "bir haftalık saatlik veri" → **lokasyon bazlı tek özet** → **tek görsel** olur; zaman boyutu haritada değil, istersen ayrı bir zaman serisi grafiğinde gösterilir.

Aşağıda **kurs verisi** (`df_week`) ile lokasyon özeti alınıp benzinlik haritasının üzerine **trafik katmanı** ekleniyor.

In [31]:
# Saatlik veriyi lokasyon başına özetle (bir hafta → her sensör için ortalama araç sayısı ve hız)
if df_week is not None:
    traffic_summary = (
        df_week.groupby(["GEOHASH", "LATITUDE", "LONGITUDE"], as_index=False)
        .agg(
            mean_vehicles=("NUMBER_OF_VEHICLES", "mean"),
            mean_speed=("AVERAGE_SPEED", "mean"),
            total_readings=("DATE_TIME", "count"),
        )
        .rename(columns={"LATITUDE": "lat", "LONGITUDE": "lon"})
    )
    print("Lokasyon özeti (ilk 5):")
    display(traffic_summary.head())
    print(f"Toplam {len(traffic_summary)} sensör noktası (haritada hepsi tek daire ile gösterilecek).")
else:
    traffic_summary = None
    print("df_week yok; önce 'istanbul_traffic_week.csv' yükleyin.")

Lokasyon özeti (ilk 5):


,GEOHASH,lat,lon,mean_vehicles,mean_speed,total_readings
0,sx7chk,40.981750,27.965698,23.100000,85.550000,20
1,sx7chm,40.987244,27.965698,39.760000,71.040000,25
2,sx7cht,40.987244,27.976685,34.720000,36.680000,25
3,sx7chw,40.992737,27.976685,33.368421,66.894737,19
4,sx7chx,40.998230,27.976685,38.619048,70.238095,21


Toplam 2439 sensör noktası (haritada hepsi tek daire ile gösterilecek).


In [32]:
# Tek harita: benzinlikler (kırmızı) + trafik özeti (daire boyutu = ortalama araç sayısı)
if traffic_summary is not None and len(traffic_summary) > 0:
    import numpy as np
    v = traffic_summary["mean_vehicles"]
    r_min, r_max = 1, 8
    traffic_summary = traffic_summary.copy()
    traffic_summary["radius"] = r_min + (r_max - r_min) * (v - v.min()) / (v.max() - v.min() + 1e-9)
    m2 = folium.Map(location=ISTANBUL_CENTER, zoom_start=10, tiles="OpenStreetMap")
    for _, row in traffic_summary.iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=float(row["radius"]),
            color="blue",
            fill=True,
            fill_color="blue",
            fill_opacity=0.25,
            weight=0.5,
            popup=f"Ort. araç: {row['mean_vehicles']:.0f}, Ort. hız: {row['mean_speed']:.1f} km/h",
        ).add_to(m2)
    for _, row in df_gas.iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=4,
            color="red",
            fill=True,
            fill_color="red",
            fill_opacity=0.7,
            popup=row["name"] or "Benzinlik",
            weight=1,
        ).add_to(m2)
    m2
else:
    print("Trafik özeti yok; yukarıdaki hücrede df_week yüklü olmalı.")